In [ ]:
from datetime import datetime

In [ ]:
import pandas as pd

# Load the data
data = pd.read_csv("M1_predictions.csv")
def is_ymd_format(date_str):
    """Check if the date string is in YYYY-MM-DD format."""
    try:
        pd.to_datetime(date_str, format='%Y-%m-%d')
        return True
    except ValueError:
        return False

# Apply the change only if the format is not already YYYY-MM-DD
data['cohort'] = data['cohort'].apply(lambda x: pd.to_datetime(x, format='%d-%m-%Y').strftime('%Y-%m-%d') 
                                      if not is_ymd_format(x) else x)




In [ ]:
data.head()

In [ ]:
# Function to sort predictions
def sort_predictions(row, prefix):
    values = sorted([row[f'{prefix}_bad'], row[f'{prefix}_expected'], row[f'{prefix}_good']])
    row[f'{prefix}_bad'], row[f'{prefix}_expected'], row[f'{prefix}_good'] = values
    return row

# Apply sorting for each group of predictions
for feature_group in ['transactional', 'cumulative', 'absolute_revenue']:
    data = data.apply(lambda row: sort_predictions(row, feature_group), axis=1)

# Save the updated data
data.to_csv("new_M1.csv", index=False)

In [ ]:
data.head()

In [ ]:
import pandas as pd
import numpy as np

# Load your original data
data = pd.read_csv("new_M1.csv")

# Step 1: If needed, add a unique identifier for each row 
# (still sometimes helpful for debugging or other merges)
data = data.reset_index().rename(columns={'index': 'row_id'})

# Step 2: Filter rows that have valid actuals and predictions
filtered_data = data[
    (data['transactional_actual'].notnull()) &
    (data['transactional_actual'] > 0) &
    (data['transactional_expected'].notnull())
].copy()

# Calculate squared error for these rows
filtered_data['squared_error'] = (
    filtered_data['transactional_actual'] - filtered_data['transactional_expected']
) ** 2

# -------------------------------------------------------------------
#  NEW APPROACH: Compute group-level metrics and then merge them back
# -------------------------------------------------------------------

# 1. Compute (game-level) model RMSE and (cohort-level) cohort RMSE
grouped_metrics = []

for game_id, game_group in filtered_data.groupby('game_id'):
    # game-level RMSE
    model_rmse = np.sqrt(game_group['squared_error'].mean())

    # cohort-level RMSE for each cohort within the same game
    cohort_rmse_df = (
        game_group.groupby('cohort')['squared_error']
        .mean()
        .apply(np.sqrt)
        .reset_index()
        .rename(columns={'squared_error': 'cohort_rmse'})
    )
    
    # Add columns for merging/tracking
    cohort_rmse_df['game_id'] = game_id
    cohort_rmse_df['model_rmse'] = model_rmse

    grouped_metrics.append(cohort_rmse_df)

# Combine into one DataFrame
grouped_metrics = pd.concat(grouped_metrics, ignore_index=True)

# 2. Compute confidence_trend per cohort (within each game)
grouped_metrics['confidence_trend'] = (
    grouped_metrics['model_rmse'] - grouped_metrics['cohort_rmse']
) / grouped_metrics['model_rmse']

# 3. Compute absolute confidence & scale it [0..1] within each game_id
grouped_metrics['abs_conf'] = grouped_metrics['confidence_trend'].abs()

def scale_values(series):
    """Scales values in a Series to [0..1]."""
    min_val = series.min()
    max_val = series.max()
    if min_val == max_val:
        # Avoid division by zero; if all values are the same, just return 0
        return np.zeros(len(series))
    return (series - min_val) / (max_val - min_val)

grouped_metrics['scaled_confidence_score'] = (
    grouped_metrics.groupby('game_id')['abs_conf'].transform(scale_values)
)

# 4. Categorize confidence
def categorize_confidence(score):
    if score <= 0.33:
        return 'High'
    elif score <= 0.67:
        return 'Moderate'
    else:
        return 'Low'

grouped_metrics['confidence_category'] = grouped_metrics[
    'scaled_confidence_score'
].apply(categorize_confidence)

# 5. Now we have (per game_id, cohort) all the columns needed:
#    'cohort_rmse', 'model_rmse', 'confidence_trend', 
#    'scaled_confidence_score', 'confidence_category'

#    Merge them back into the ENTIRE dataset (not just filtered!)
data = data.merge(
    grouped_metrics[[
        'game_id',
        'cohort',
        'cohort_rmse',
        'model_rmse',
        'confidence_trend',
        'scaled_confidence_score',
        'confidence_category'
    ]],
    on=['game_id', 'cohort'],
    how='left'
)

# Now, all rows that share the same (game_id, cohort)—including those
# that only had a predicted value—will receive the same confidence
# metrics computed from the rows that *did* have actuals.

# You can drop row_id if you no longer need it:
data.drop(columns=['row_id'], inplace=True, errors='ignore')

# data now contains your original columns plus:
#   'cohort_rmse', 'model_rmse', 'confidence_trend', 
#   'scaled_confidence_score', 'confidence_category'


In [ ]:
new_column_names = {
    'current_month': 'transaction_period',
    'transactional_actual': 'incremental_net_roas',
    'cumulative_actual': 'cumulative_net_roas',
    'absolute_revenue_actual': 'net_revenue',
    'transactional_bad': 'incremental_net_roas_bad_pred',
    'transactional_expected': 'incremental_net_roas_expected_pred',
    'transactional_good': 'incremental_net_roas_good_pred',
    'cumulative_bad': 'cumulative_net_roas_bad',
    'cumulative_expected':'cumulative_net_roas_expected',
    'cumulative_good':'cumulative_net_roas_good',

}
data.rename(columns=new_column_names, inplace=True)

data['cohort'] = pd.to_datetime(data['cohort']).dt.strftime('%Y-%m')

today_date = datetime.now().strftime('%Y-%m')
data['created_at'] = today_date
data['updated_at'] = today_date

In [ ]:
current_year_month = datetime.now().strftime('%Y-%m')
cols_to_nullify = ['incremental_net_roas', 'cumulative_net_roas', 'net_revenue']
data.loc[data['transaction_period'] >= current_year_month, cols_to_nullify] = np.nan

In [ ]:
import numpy as np
import pandas as pd

# Ensure data is sorted properly by game_id, cohort, and elapsed_months
data = data.sort_values(by=['game_id', 'cohort', 'elapsed_months']).reset_index(drop=True)

# Initialize new cumulative columns
data['cumulative_net_roas_bad_pred'] = np.nan
data['cumulative_net_roas_expected_pred'] = np.nan
data['cumulative_net_roas_good_pred'] = np.nan

# Group by game_id and cohort to ensure tracking is independent for each (game, cohort) pair
for (game_id, cohort), game_group in data.groupby(['game_id', 'cohort']):
    # Initialize cumulative values for the 3 categories
    cumulative_net_roas_bad = None
    cumulative_net_roas_expected = None
    cumulative_net_roas_good = None
    
    for i, row_index in enumerate(game_group.index):
        row = data.loc[row_index]
        
        if row['elapsed_months'] == 1:
            # For the first month, set the cumulative to the actual cumulative for all categories
            cumulative_net_roas_bad = row['cumulative_net_roas']
            cumulative_net_roas_expected = row['cumulative_net_roas']
            cumulative_net_roas_good = row['cumulative_net_roas']
        elif(not np.isnan(x)):
            # For months > 1, always add to the previous cumulative predictions
            cumulative_net_roas_bad = x+ row['incremental_net_roas_bad_pred']
            cumulative_net_roas_expected = x+row['incremental_net_roas_expected_pred']
            cumulative_net_roas_good = x+row['incremental_net_roas_good_pred']

        else:
            cumulative_net_roas_bad += row['incremental_net_roas_bad_pred']
            cumulative_net_roas_expected += row['incremental_net_roas_expected_pred']
            cumulative_net_roas_good += row['incremental_net_roas_good_pred']


        x=row['cumulative_net_roas']
        
        # Store the calculated cumulative values in the DataFrame
        data.loc[row_index, 'cumulative_net_roas_bad_pred'] = cumulative_net_roas_bad
        data.loc[row_index, 'cumulative_net_roas_expected_pred'] = cumulative_net_roas_expected
        data.loc[row_index, 'cumulative_net_roas_good_pred'] = cumulative_net_roas_good


In [ ]:
# Create 3 new cumulative columns for "WA" logic
data['cumulative_net_roas_bad_wa'] = np.nan
data['cumulative_net_roas_expected_wa'] = np.nan
data['cumulative_net_roas_good_wa'] = np.nan

# Group by game_id and cohort to apply the logic independently for each group
for (game_id, cohort), game_group in data.groupby(['game_id', 'cohort']):
    for row_index in game_group.index:
        row = data.loc[row_index]
        
        if pd.notnull(row['cumulative_net_roas']):
            # If cumulative_net_roas is available, use it for all 3 WA columns
            data.loc[row_index, 'cumulative_net_roas_bad_wa'] = row['cumulative_net_roas']
            data.loc[row_index, 'cumulative_net_roas_expected_wa'] = row['cumulative_net_roas']
            data.loc[row_index, 'cumulative_net_roas_good_wa'] = row['cumulative_net_roas']
        else:
            # If cumulative_net_roas is NaN, use the calculated cumulative predictions
            data.loc[row_index, 'cumulative_net_roas_bad_wa'] = row['cumulative_net_roas_bad_pred']
            data.loc[row_index, 'cumulative_net_roas_expected_wa'] = row['cumulative_net_roas_expected_pred']
            data.loc[row_index, 'cumulative_net_roas_good_wa'] = row['cumulative_net_roas_good_pred']


In [ ]:
data.head(28)

In [ ]:
# Step 1: Drop the unnecessary columns
columns_to_drop = ['squared_error', 'cohort_rmse', 'model_rmse','confidence_trend']
data = data.drop(columns=columns_to_drop, errors='ignore')

# Step 2: Rename columns containing 'absolute_revenue' to 'net_revenue'
data = data.rename(columns=lambda col: col.replace('absolute_revenue', 'net_revenue'))

# Display the updated column names and their data types
data_types_df = pd.DataFrame({
    'Column Name': data.columns,
    'Data Type': data.dtypes.values
})

# Print the updated DataFrame with column names and data types
print(data_types_df)



In [ ]:
# # List of specific columns to convert to datetime
# columns_to_convert = ['cohort', 'transaction_period', 'created_at', 'updated_at']

# # Convert the specified columns to datetime
# for col in columns_to_convert:
#     if col in data.columns:  # Ensure the column exists in the DataFrame
#         data[col] = pd.to_datetime(data[col], errors='coerce')
#         # Print a message indicating success or failure
#         if pd.api.types.is_datetime64_any_dtype(data[col]):
#             print(f"✅ Column '{col}' successfully converted to datetime")
#         else:
#             print(f"❌ Column '{col}' could not be converted to datetime")

# # Print the updated data types of all columns
# data_types_df = pd.DataFrame({
#     'Column Name': data.columns,
#     'Data Type': data.dtypes.values
# })

# # Print the updated data types
# print(data_types_df)


In [ ]:
data.head()

In [ ]:
# Save the results to a new CSV file
data.to_csv("final_M1.csv", index=False)

In [ ]:
g1 = data[data['game_id']=='G001']
min(g1['incremental_net_roas_bad_pred'])